# torch

## Vision Transformer from Scratch

First, let's define a Vision Transformer model class in PyTorch. We'll make a small ViT with configurable parameters. We need to implement the patch embedding, class token, positional embedding, and a Transformer encoder. PyTorch has a built-in nn.TransformerEncoder and nn.MultiheadAttention which we can use to simplify things. Here's the code:

In [1]:
import torch
import torch.nn as nn
import math
class VisionTransformer(nn.Module):
    def __init__(self, image_size=32, patch_size=8, num_classes=10,
                 dim=128, depth=4, heads=4, mlp_dim=256, channels=3, dropout=0.1):
        super(VisionTransformer, self).__init__()
        assert image_size % patch_size == 0, "Image dimensions must be divisible by patch size."

        # Number of patches
        num_patches = (image_size // patch_size) ** 2
        patch_dim = channels * patch_size * patch_size  # number of input features per patch

        # Patch embedding: linear projection of flattened patches to model dimension (dim)
        self.patch_to_emb = nn.Linear(patch_dim, dim)
        # Class token: a learnable embedding that represents the whole image (added to sequence)
        self.class_token = nn.Parameter(torch.randn(1, 1, dim))
        # Positional embedding: learnable embeddings for each patch position + 1 class token
        self.positional_emb = nn.Parameter(torch.randn(1, num_patches + 1, dim))

        # Transformer encoder layers
        encoder_layer = nn.TransformerEncoderLayer(d_model=dim, nhead=heads,
                                                  dim_feedforward=mlp_dim, dropout=dropout, activation='gelu',
                                                  batch_first=True)  # batch_first allows (batch, seq, dim) input
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        # Classification head (MLP Head)
        self.to_cls_token = nn.Identity()  # identity (not needed, but for clarity)
        self.mlp_head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, num_classes)
        )

    def forward(self, x):
        """
        x: input images of shape (batch, channels, height, width)
        """
        B, C, H, W = x.shape
        # 1. Divide image into patches and flatten
        # Suppose image is (B, C, 32, 32) and patch_size=8 -> 4 patches along each dimension -> 16 patches total
        # We can reshape and permute to get patches
        patch_size = int(H // (H / (H // (H if H==W else 1))))  # quick way to get patch_size from H (here just use given patch_size)
        # Actually, better to use the patch_size passed (assuming H == W == image_size)
        patch_size = int(math.sqrt(self.patch_to_emb.in_features / C)) if hasattr(self, 'patch_to_emb') else patch_size

        # Reshape into patches
        x = x.reshape(B, C, H//patch_size, patch_size, W//patch_size, patch_size)
        x = x.permute(0, 2, 4, 1, 3, 5)  # (B, n_patches_h, n_patches_w, C, patch_size, patch_size)
        patches = x.reshape(B, -1, C * patch_size * patch_size)  # (B, num_patches, patch_dim)

        # 2. Linear projection to get patch embeddings
        patch_embeddings = self.patch_to_emb(patches)  # (B, num_patches, dim)

        # 3. Prepend class token to patch embeddings
        cls_tokens = self.class_token.expand(B, -1, -1)  # (B, 1, dim)
        x = torch.cat([cls_tokens, patch_embeddings], dim=1)  # (B, 1 + num_patches, dim)

        # 4. Add positional embeddings
        x = x + self.positional_emb[:, :x.size(1), :]

        # 5. Pass through Transformer encoder
        x = self.transformer(x)  # (B, 1+num_patches, dim)

        # 6. Take the class token output and apply the classification head
        cls_output = x[:, 0]  # (B, dim) -> the [CLS] token's representation
        out = self.mlp_head(cls_output)  # (B, num_classes)
        return out
# Instantiate the model for CIFAR-10 (32x32 images, say patch size 4 so we get 8x8=64 patches)
model = VisionTransformer(image_size=32, patch_size=4, num_classes=10, dim=128, depth=4, heads=4, mlp_dim=256)



In [2]:
# Load cifar 10
import torchvision
import torchvision.transforms as T

train_transform = T.Compose([
    T.ToTensor()
])
test_transform = T.Compose([
    T.ToTensor()
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")


100%|██████████| 170M/170M [00:04<00:00, 39.8MB/s]


Train batches: 782, Test batches: 157


In [3]:
# train VIT
import torch.optim as optim
# Move model to device (GPU if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
epochs = 10
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)            # forward pass
        loss = criterion(outputs, labels)  # compute loss
        loss.backward()                    # backpropagate gradients
        optimizer.step()                   # update parameters

        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

Epoch [1/10], Loss: 1.9978
Epoch [2/10], Loss: 1.7978
Epoch [3/10], Loss: 1.6973
Epoch [4/10], Loss: 1.6171
Epoch [5/10], Loss: 1.5608
Epoch [6/10], Loss: 1.5168
Epoch [7/10], Loss: 1.4839
Epoch [8/10], Loss: 1.4492
Epoch [9/10], Loss: 1.4204
Epoch [10/10], Loss: 1.3920


In [4]:
# Evaluation
model.eval()
correct = 0
total = 0
with torch.no_grad():  # no gradient needed for eval
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)  # outputs are of shape (batch, 10)
        _, predicted = torch.max(outputs, dim=1)  # class with highest score
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 50.86%


# numpy version v0

In [7]:
# import numpy as np

# class VisionTransformerNumpy:
#     def __init__(self, image_size=32, patch_size=8, num_classes=10,
#                  dim=128, depth=4, heads=4, mlp_dim=256, channels=3, dropout=0.1):
#         assert image_size % patch_size == 0, "Image dimensions must be divisible by patch size."

#         self.image_size = image_size
#         self.patch_size = patch_size
#         self.num_patches = (image_size // patch_size) ** 2
#         self.patch_dim = channels * patch_size * patch_size
#         self.dim = dim
#         self.depth = depth
#         self.heads = heads
#         self.mlp_dim = mlp_dim
#         self.channels = channels

#         # Weights for patch embedding
#         self.patch_to_emb_W = np.random.randn(self.patch_dim, dim) * 0.02
#         self.patch_to_emb_b = np.zeros((dim,))

#         # Class token
#         self.class_token = np.random.randn(1, dim)

#         # Positional embedding
#         self.positional_emb = np.random.randn(1 + self.num_patches, dim)

#         # Transformer weights (simplified, shared for demonstration)
#         self.attn_weights_Q = np.random.randn(dim, dim)
#         self.attn_weights_K = np.random.randn(dim, dim)
#         self.attn_weights_V = np.random.randn(dim, dim)
#         self.attn_out = np.random.randn(dim, dim)

#         # Feedforward (MLP) layer weights
#         self.ffn_w1 = np.random.randn(dim, mlp_dim)
#         self.ffn_b1 = np.zeros((mlp_dim,))
#         self.ffn_w2 = np.random.randn(mlp_dim, dim)
#         self.ffn_b2 = np.zeros((dim,))

#         # Classification head
#         self.head_w = np.random.randn(dim, num_classes)
#         self.head_b = np.zeros((num_classes,))

#     def softmax(self, x, axis=-1):
#         e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
#         return e_x / np.sum(e_x, axis=axis, keepdims=True)

#     def layer_norm(self, x, eps=1e-6):
#         mean = np.mean(x, axis=-1, keepdims=True)
#         std = np.std(x, axis=-1, keepdims=True)
#         return (x - mean) / (std + eps)

#     def attention(self, x):
#         Q = x @ self.attn_weights_Q
#         K = x @ self.attn_weights_K
#         V = x @ self.attn_weights_V
#         scores = (Q @ K.transpose(0, 2, 1)) / np.sqrt(self.dim)
#         weights = self.softmax(scores, axis=-1)
#         out = weights @ V
#         out = out @ self.attn_out
#         return out

#     def feed_forward(self, x):
#         x = x @ self.ffn_w1 + self.ffn_b1
#         x = np.maximum(0, x)  # ReLU (can replace with GELU if desired)
#         x = x @ self.ffn_w2 + self.ffn_b2
#         return x

#     def transformer_block(self, x):
#         # Multi-head self-attention + residual
#         attn_out = self.attention(self.layer_norm(x))
#         x = x + attn_out

#         # Feedforward + residual
#         ff_out = self.feed_forward(self.layer_norm(x))
#         x = x + ff_out
#         return x

#     def forward(self, images , return_cls_output=False):
#         B, C, H, W = images.shape
#         P = self.patch_size
#         # Split image into patches
#         patches = images.reshape(B, C, H // P, P, W // P, P)
#         patches = patches.transpose(0, 2, 4, 1, 3, 5)
#         patches = patches.reshape(B, -1, self.patch_dim)

#         # Linear embedding
#         patch_embeddings = patches @ self.patch_to_emb_W + self.patch_to_emb_b

#         # Add class token
#         cls_tokens = np.tile(self.class_token, (B, 1)).reshape(B, 1, -1)
#         x = np.concatenate([cls_tokens, patch_embeddings], axis=1)

#         # Add positional embeddings
#         x = x + self.positional_emb[:x.shape[1]]

#         # Transformer blocks
#         for _ in range(self.depth):
#             x = self.transformer_block(x)

#         # Classification head
#         cls_output = x[:, 0]  # take [CLS] token
#         if return_cls_output:
#           return cls_output
#         logits = cls_output @ self.head_w + self.head_b
#         return logits

# # Example input image batch: (batch_size, channels, height, width)
# np.random.seed(42)
# images = np.random.randn(8, 3, 32, 32)  # 8 CIFAR-10 images
# vit = VisionTransformerNumpy(image_size=32, patch_size=4, num_classes=10)
# output = vit.forward(images)
# print("Output logits shape:", output.shape)  # Should be (8, 10)


Output logits shape: (8, 10)


In [8]:
# import os
# import urllib.request
# import tarfile
# import pickle
# import numpy as np

# def download_and_extract_cifar10(destination="./data"):
#     url = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"
#     filename = url.split("/")[-1]
#     filepath = os.path.join(destination, filename)
#     extracted_path = os.path.join(destination, "cifar-10-batches-py")

#     os.makedirs(destination, exist_ok=True)

#     if not os.path.exists(extracted_path):
#         print("Downloading CIFAR-10...")
#         urllib.request.urlretrieve(url, filepath)
#         print("Extracting...")
#         with tarfile.open(filepath, "r:gz") as tar:
#             tar.extractall(path=destination)
#         print("Done.")
#     return extracted_path

# def load_cifar10_batch(file):
#     with open(file, 'rb') as fo:
#         dict = pickle.load(fo, encoding='bytes')
#         data = dict[b'data']  # shape: (10000, 3072)
#         labels = dict[b'labels']
#         data = data.reshape(-1, 3, 32, 32).astype(np.float32) / 255.0  # Normalize to [0,1]
#         labels = np.array(labels)
#         return data, labels

# def load_cifar10_dataset(path):
#     train_data = []
#     train_labels = []
#     for i in range(1, 6):
#         batch_path = os.path.join(path, f"data_batch_{i}")
#         data, labels = load_cifar10_batch(batch_path)
#         train_data.append(data)
#         train_labels.append(labels)
#     X_train = np.concatenate(train_data)
#     y_train = np.concatenate(train_labels)

#     X_test, y_test = load_cifar10_batch(os.path.join(path, "test_batch"))
#     return (X_train, y_train), (X_test, y_test)

# # Create batches manually
# def create_batches(X, y, batch_size, shuffle=True):
#     indices = np.arange(len(X))
#     if shuffle:
#         np.random.shuffle(indices)
#     for start_idx in range(0, len(X), batch_size):
#         end_idx = start_idx + batch_size
#         batch_idx = indices[start_idx:end_idx]
#         yield X[batch_idx], y[batch_idx]

# # Usage
# cifar_path = download_and_extract_cifar10()
# (X_train, y_train), (X_test, y_test) = load_cifar10_dataset(cifar_path)

# # Create "loaders"
# train_loader = list(create_batches(X_train, y_train, batch_size=64, shuffle=True))
# test_loader = list(create_batches(X_test, y_test, batch_size=64, shuffle=False))

# print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")


Train batches: 782, Test batches: 157


In [9]:
# import numpy as np

# def cross_entropy_loss(logits, labels):
#     """
#     logits: (batch_size, num_classes)
#     labels: (batch_size,) with class indices
#     """
#     logits = logits - np.max(logits, axis=1, keepdims=True)  # stability
#     exp_logits = np.exp(logits)
#     probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
#     N = logits.shape[0]
#     log_likelihood = -np.log(probs[range(N), labels])
#     loss = np.sum(log_likelihood) / N
#     return loss

# def compute_accuracy(logits, labels):
#     preds = np.argmax(logits, axis=1)
#     return np.mean(preds == labels)

# # Dummy backward + update (manual SGD for classification head only)
# def sgd_update_class_head(vit, logits, labels, learning_rate):
#     N = logits.shape[0]
#     probs = np.exp(logits - np.max(logits, axis=1, keepdims=True))
#     probs /= np.sum(probs, axis=1, keepdims=True)
#     probs[range(N), labels] -= 1  # softmax gradient

#     dlogits = probs / N  # (batch_size, num_classes)
#     dW = vit.cls_output.T @ dlogits  # (dim, num_classes)
#     db = np.sum(dlogits, axis=0)

#     vit.head_w -= learning_rate * dW
#     vit.head_b -= learning_rate * db

# def train_vit_numpy(vit, train_loader, epochs=10, lr=1e-3):
#     for epoch in range(epochs):
#         running_loss = 0
#         acc = 0
#         for images, labels in train_loader:
#             logits = vit.forward(images)
#             loss = cross_entropy_loss(logits, labels)
#             acc += compute_accuracy(logits, labels)

#             # Backprop + update only final linear layer (for demo)
#             vit.cls_output = vit.forward(images, return_cls_output=True)
#             sgd_update_class_head(vit, logits, labels, learning_rate=lr)

#             running_loss += loss

#         avg_loss = running_loss / len(train_loader)
#         avg_acc = acc / len(train_loader)
#         print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}, Acc: {avg_acc:.4f}")


In [10]:
# train_vit_numpy(vit, train_loader, epochs=10, lr=1e-3)


<ipython-input-9-3bbbcd0e79c9>:12: RuntimeWarning: divide by zero encountered in log
  log_likelihood = -np.log(probs[range(N), labels])


Epoch [1/10], Loss: inf, Acc: 0.1094


KeyboardInterrupt: 

In [ ]:
# def evaluate_vit_numpy(vit, test_loader):
#     correct = 0
#     total = 0

#     for images, labels in test_loader:
#         logits = vit.forward(images)  # shape: (batch, num_classes)
#         predictions = np.argmax(logits, axis=1)
#         correct += np.sum(predictions == labels)
#         total += len(labels)

#     accuracy = 100.0 * correct / total
#     print(f"Test Accuracy: {accuracy:.2f}%")

# evaluate_vit_numpy(vit, test_loader)



# numpy version 1

In [17]:
!wget https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz
!tar -xvzf cifar-10-python.tar.gz


--2025-04-14 20:11:26--  https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz
Resolving www.cs.toronto.edu (www.cs.toronto.edu)... 128.100.3.30
Connecting to www.cs.toronto.edu (www.cs.toronto.edu)|128.100.3.30|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 170498071 (163M) [application/x-gzip]
Saving to: ‘cifar-10-python.tar.gz’

cifar-10-python.tar 100%[===================>] 162.60M  28.2MB/s    in 7.9s    

2025-04-14 20:11:34 (20.5 MB/s) - ‘cifar-10-python.tar.gz’ saved [170498071/170498071]

cifar-10-batches-py/
cifar-10-batches-py/data_batch_4
cifar-10-batches-py/readme.html
cifar-10-batches-py/test_batch
cifar-10-batches-py/data_batch_3
cifar-10-batches-py/batches.meta
cifar-10-batches-py/data_batch_2
cifar-10-batches-py/data_batch_5
cifar-10-batches-py/data_batch_1


In [30]:
import pickle
import numpy as np
import os
import matplotlib.pyplot as plt

def load_cifar10_batch(batch_path):
    with open(batch_path, 'rb') as f:
        batch = pickle.load(f, encoding='bytes')
        data = batch[b'data']  # shape (10000, 3072)
        labels = batch[b'labels']
        data = data.reshape(-1, 3, 32, 32).astype(np.float32) / 255.0
        labels = np.array(labels)
    return data, labels

def load_cifar10_numpy(data_dir='./cifar-10-batches-py'):
    xs, ys = [], []
    for i in range(1, 6):
        X, Y = load_cifar10_batch(os.path.join(data_dir, f'data_batch_{i}'))
        xs.append(X)
        ys.append(Y)
    X_train, Y_train = np.concatenate(xs), np.concatenate(ys)
    X_test, Y_test = load_cifar10_batch(os.path.join(data_dir, 'test_batch'))
    return (X_train, Y_train), (X_test, Y_test)


In [31]:
import numpy as np

class VisionTransformerNumpy:
    def __init__(self, image_size=32, patch_size=4, num_classes=10, dim=128, channels=3):
        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        self.patch_dim = channels * patch_size * patch_size
        self.dim = dim

        # Patch embedding
        self.W_patch = np.random.randn(self.patch_dim, dim) * 0.02

        # Class token and positional embedding
        self.class_token = np.random.randn(1, dim)
        self.pos_emb = np.random.randn(self.num_patches + 1, dim)  # shape (num_patches+1, dim)

        # Classification head
        self.head_w = np.random.randn(dim, num_classes) * 0.02
        self.head_b = np.zeros(num_classes)

    def extract_patches(self, x):
        B, C, H, W = x.shape
        ps = self.patch_size
        x = x.reshape(B, C, H // ps, ps, W // ps, ps)
        x = x.transpose(0, 2, 4, 1, 3, 5)  # (B, num_patches_h, num_patches_w, C, ps, ps)
        patches = x.reshape(B, -1, self.patch_dim)
        return patches

    def forward(self, x, return_cls_output=False):
        patches = self.extract_patches(x)
        B = x.shape[0]

        # Patch embedding
        x = patches @ self.W_patch  # (B, num_patches, dim)

        # Add class token
        cls_tokens = np.tile(self.class_token, (B, 1)).reshape(B, 1, -1)  # (B, 1, dim)
        x = np.concatenate([cls_tokens, x], axis=1)  # (B, num_patches + 1, dim)

        # Add positional embeddings
        x += self.pos_emb[:x.shape[1]]  # broadcasting (num_patches+1, dim)

        # Classification head
        cls_output = x[:, 0]  # (B, dim)
        if return_cls_output:
            return cls_output
        logits = cls_output @ self.head_w + self.head_b  # (B, num_classes)
        return logits


In [32]:
def batch_loader(X, Y, batch_size=64, shuffle=True):
    indices = np.arange(len(X))
    if shuffle:
        np.random.shuffle(indices)
    for start in range(0, len(X), batch_size):
        batch_idx = indices[start:start + batch_size]
        yield X[batch_idx], Y[batch_idx]


In [33]:
def cross_entropy_loss(logits, labels):
    logits = logits - np.max(logits, axis=1, keepdims=True)
    probs = np.exp(logits)
    probs /= np.sum(probs, axis=1, keepdims=True)
    N = logits.shape[0]
    return -np.mean(np.log(probs[np.arange(N), labels]))

def compute_accuracy(logits, labels):
    preds = np.argmax(logits, axis=1)
    return np.mean(preds == labels)

def update_classifier(vit, cls_output, logits, labels, lr=1e-3):
    N = logits.shape[0]
    probs = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    probs /= np.sum(probs, axis=1, keepdims=True)
    probs[np.arange(N), labels] -= 1
    probs /= N
    grad_w = cls_output.T @ probs
    grad_b = np.sum(probs, axis=0)
    vit.head_w -= lr * grad_w
    vit.head_b -= lr * grad_b


In [34]:
def train_numpy_vit(vit, X_train, Y_train, epochs=10, lr=1e-3, batch_size=64):
    for epoch in range(epochs):
        losses, accs = [], []
        for x_batch, y_batch in batch_loader(X_train, Y_train, batch_size):
            cls_out = vit.forward(x_batch, return_cls_output=True)
            logits = cls_out @ vit.head_w + vit.head_b
            loss = cross_entropy_loss(logits, y_batch)
            acc = compute_accuracy(logits, y_batch)
            update_classifier(vit, cls_out, logits, y_batch, lr)
            losses.append(loss)
            accs.append(acc)
        print(f"Epoch {epoch+1}: Loss={np.mean(losses):.4f}, Accuracy={np.mean(accs)*100:.2f}%")

def evaluate_numpy_vit(vit, X_test, Y_test, batch_size=64):
    total_correct = 0
    total = 0
    for x_batch, y_batch in batch_loader(X_test, Y_test, batch_size, shuffle=False):
        logits = vit.forward(x_batch)
        preds = np.argmax(logits, axis=1)
        total_correct += np.sum(preds == y_batch)
        total += len(y_batch)
    print(f"Test Accuracy: {100 * total_correct / total:.2f}%")


In [35]:
(X_train, Y_train), (X_test, Y_test) = load_cifar10_numpy()

vit = VisionTransformerNumpy(image_size=32, patch_size=4, num_classes=10, dim=128)

train_numpy_vit(vit, X_train, Y_train, epochs=10, lr=1e-3)
evaluate_numpy_vit(vit, X_test, Y_test)


Epoch 1: Loss=2.3045, Accuracy=9.79%
Epoch 2: Loss=2.3034, Accuracy=10.15%
Epoch 3: Loss=2.3035, Accuracy=9.88%
Epoch 4: Loss=2.3035, Accuracy=9.83%
Epoch 5: Loss=2.3033, Accuracy=10.08%
Epoch 6: Loss=2.3034, Accuracy=9.93%
Epoch 7: Loss=2.3034, Accuracy=9.91%
Epoch 8: Loss=2.3035, Accuracy=9.76%
Epoch 9: Loss=2.3035, Accuracy=10.02%
Epoch 10: Loss=2.3035, Accuracy=9.81%
Test Accuracy: 10.00%


In [36]:
def show_predictions(vit, X, Y, class_names, num=5):
    logits = vit.forward(X[:num])
    preds = np.argmax(logits, axis=1)
    for i in range(num):
        plt.imshow(np.transpose(X[i], (1, 2, 0)))
        plt.title(f"Label: {class_names[Y[i]]}, Pred: {class_names[preds[i]]}")
        plt.axis('off')
        plt.show()
